# 05 — Construindo um dataset astronômico multimodal

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** como representar o mesmo objeto usando astrometria, fotometria, espectroscopia e imagem sem perder proveniência?

Este notebook fecha o workshop. Em vez de treinar imediatamente uma rede neural, construímos o produto que normalmente determina a qualidade do modelo: uma tabela de treinamento com identificadores, medidas, qualidade, separações de crossmatch e referências para produtos de imagem.

A ideia é mostrar AI-Scope como uma camada de preparação de dados para IA, e não apenas como um lugar para executar queries.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"
USERNAME = os.getenv("AI_SCOPE_USERNAME")
PASSWORD = os.getenv("AI_SCOPE_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("Defina AI_SCOPE_USERNAME e AI_SCOPE_PASSWORD no ambiente.")

cl = adss.ADSSClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)

In [ ]:
# Exemplo: uma lista científica pequena que poderia vir de qualquer notebook anterior.
objects = pd.DataFrame({
    "object_id": ["obj_001", "obj_002", "obj_003"],
    "ra": [150.10, 150.20, 150.30],
    "dec": [2.10, 2.20, 2.30],
})

In [ ]:
query = """
SELECT
    u.object_id,
    u.ra AS input_ra,
    u.dec AS input_dec,
    g.source_id AS gaia_source_id,
    g.phot_g_mean_mag,
    g.phot_bp_mean_mag,
    g.phot_rp_mean_mag,
    g.parallax,
    g.pmra,
    g.pmdec,
    g.ruwe,
    g.teff_gspphot,
    g.logg_gspphot,
    g.mh_gspphot,
    dist(u.ra, u.dec, g.ra, g.dec) AS gaia_sep_deg
FROM upload.objects AS u
JOIN public.gaia_dr3 AS g
  ON cone(g.ra, g.dec, u.ra, u.dec, 0.0003)
"""

gaia_match = cl.query_and_wait(
    query_text=query,
    mode="astroql",
    file=objects,
    table_name="objects",
).data

gaia_match["gaia_sep_arcsec"] = gaia_match.gaia_sep_deg * 3600
gaia_match.head()

In [ ]:
# Segunda modalidade: espectroscopia DESI para os mesmos objetos.
query = """
SELECT
    u.object_id,
    d.id AS desi_id,
    d.z, d.zerr, d.zwarn,
    d.spectype, d.subtype,
    d.deltachi2,
    dist(u.ra, u.dec, d.ra, d.dec) AS desi_sep_deg
FROM upload.objects AS u
JOIN desi.desi_specz_dr1 AS d
  ON cone(d.ra, d.dec, u.ra, u.dec, 0.0003)
"""

desi_match = cl.query_and_wait(
    query_text=query,
    mode="astroql",
    file=objects,
    table_name="objects",
).data

desi_match["desi_sep_arcsec"] = desi_match.desi_sep_deg * 3600

In [ ]:
# Manifest multimodal: uma linha por objeto, com proveniência explícita.
manifest = objects.merge(gaia_match, on="object_id", how="left", suffixes=("", "_gaia"))
manifest = manifest.merge(desi_match, on="object_id", how="left")

# Exemplo de flags úteis antes de qualquer ML.
manifest["good_gaia_astrometry"] = manifest.ruwe < 1.4
manifest["secure_desi_redshift"] = (manifest.zwarn == 0) & manifest.z.notna()
manifest

### Desafio final: *active learning*

Imagine que apenas 5% dos objetos possuem rótulos confiáveis. Em vez de rotular uma amostra aleatória, treine um modelo inicial e escolha para inspeção os objetos de maior incerteza ou mais distantes do conjunto de treinamento. Para cada candidato, o AI-Scope pode fornecer catálogo, espectroscopia e cutout. O ciclo torna-se:

**query → modelo → incerteza → inspeção → novo rótulo → novo modelo**.

Esse é um exemplo de workflow em que a infraestrutura de dados participa diretamente do processo de descoberta científica.